# 教程: 通过配置文件运行仿真

**相关文件:**
- `run_from_config.py`
- `common/config_parser.py`
- `examples/full_case_study/config.yaml`

## 目的
对于不熟悉Python编程的用户，或者为了实现可重复、可配置的批量化模拟，通过配置文件来定义和运行模型是最高效的方式。本框架支持通过一个YAML格式的配置文件来完整地定义一个复杂的、耦合的模型网络。

此笔记本将：
1.  详细解析YAML配置文件的结构和语法。
2.  解释`ConfigParser`是如何读取YAML文件，并自动构建出相应的组件和网络。
3.  演示如何使用`run_from_config.py`脚本来执行一个在配置文件中定义的完整案例研究。
4.  加载并可视化仿真结果，展示一个完整的“免代码”工作流。

## 1. YAML配置文件结构解析

我们以`examples/full_case_study/config.yaml`为例。一个配置文件主要由四个顶级部分组成：

1.  **`simulation_parameters`**: 定义全局的仿真参数，如时间步长`dt_seconds`和总步数`num_steps`。
2.  **`components`**: **核心部分**。这是一个列表，其中每一项都定义了一个模型组件。每个组件都需要一个唯一的`name`和一个`type`（必须与代码中的类名完全匹配）。`parameters`块则包含了初始化该组件所需的所有参数。注意，这里可以进行嵌套定义，例如`HydrologicalModel`的参数中又包含了`runoff_module`和`routing_module`两个子组件。
3.  **`network`**: 定义组件之间的连接关系。每一项都指定了`from`（上游组件名称）和`to`（下游组件名称）。
4.  **`global_inputs`**: 定义全局的输入数据，如降雨、蒸发等。可以从外部文件（如CSV）加载，也可以直接在配置文件中提供一个值的列表。

下面是该配置文件的内容预览：

In [ ]:
!pygmentize ../examples/full_case_study/config.yaml

## 2. `ConfigParser` 工作原理

`common/config_parser.py`中的`ConfigParser`类是这个工作流的“翻译官”。它的`build_simulation`方法执行以下操作：
1.  **读取和解析YAML**: 使用`PyYAML`库加载配置文件。
2.  **组件工厂**: 内部维护一个“工厂”字典，将配置文件中的类型字符串（如`"HydraulicModel"`）映射到实际的Python类（`preissmann_model.model.HydraulicModel`）。
3.  **递归实例化**: 遍历`components`列表。对于每个组件，它会递归地实例化其所需的所有子组件和参数，然后实例化该组件本身。
4.  **构建网络**: 创建一个`SimulationController`实例，将所有实例化的组件添加进去，并根据`network`部分建立连接。
5.  **加载输入**: 根据`global_inputs`部分，从文件或值列表中加载数据。
6.  **返回**: 最终返回一个配置完成的`controller`对象和相关的仿真参数及全局输入，这些可以直接用于运行。

## 3. 从命令行运行

`run_from_config.py`脚本封装了上述过程。它接受一个命令行参数，即配置文件的路径。我们可以在notebook中使用`!`操作符来执行这个命令行脚本。

**注意**: 该脚本只打印每个时间步的状态，不会保存详细的过程数据或绘图。我们将在下一步手动实现这一点。

In [ ]:
!python ../run_from_config.py ../examples/full_case_study/config.yaml

## 4. 在Notebook中运行并可视化结果

为了展示一个完整的端到端工作流，我们现在直接在notebook中调用`ConfigParser`，运行仿真，并绘制最终出口的流量过程线。

In [ ]:
import sys, os
import matplotlib.pyplot as plt
import numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from common.config_parser import ConfigParser

config_file = '../examples/full_case_study/config.yaml'

print("--- 1. 从 {config_file} 构建仿真 ---")
parser = ConfigParser(config_file)
controller, sim_params, global_inputs = parser.build_simulation()

dt = sim_params.get('dt_seconds', 60)
num_steps = sim_params.get('num_steps', 1)

print("\n--- 2. 运行仿真并记录历史 ---")
history = []
history_generator = controller.run(
    num_steps=num_steps,
    dt=dt,
    global_inputs=global_inputs
)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        current_step_history[name] = {'outflow': comp.get_outflow()}
    history.append(current_step_history)

print("\n--- 3. 可视化结果 ---")
final_outlet_name = 'downstream_river'
final_outflow = [h[final_outlet_name]['outflow'] for h in history]
timesteps = np.arange(num_steps) * dt / 3600 # hours

plt.figure(figsize=(12, 6))
plt.plot(timesteps, final_outflow, 'b-o', label=f'Outflow from {final_outlet_name}')
plt.title('Case Study Simulation Results')
plt.xlabel('Time (hours)')
plt.ylabel('Flow (m³/s)')
plt.legend()
plt.grid(True)
plt.show()